# RAG-Powered PowerPoint Generator


This notebook uses **Retrieval Augmented Generation (RAG)** to:
1. Load and index the outputs from your two excel workbooks
2. Query an LLM (Gemini) with targeted questions about those outputs
3. Automatically generate a professional `.pptx` presentation from the retrieved insights

### Pipeline Overview
```
Analysis Notebook Outputs
        │
        ▼
  Text Extraction & Chunking  (LlamaIndex / SentenceSplitter)
        │
        ▼
  Embedding & Vector Store    (HuggingFace + ChromaDB)
        │
        ▼
  RAG Query Engine            (Gemini 2.5 Flash)
        │
        ▼
  Slide Content Generation    (Structured JSON via Gemini)
        │
        ▼
  PowerPoint Assembly         (python-pptx)
```

## Step 1: Install Dependencies

In [1]:
# Core RAG stack
!pip install llama-index llama-index-core llama-index-readers-file -q
!pip install llama-index-embeddings-huggingface -q
!pip install llama-index-llms-gemini -q
!pip install chromadb llama-index-vector-stores-chroma -q
!pip install llama-index-retrievers-bm25 -q # Install BM25 Retriever package
# PowerPoint generation
!pip install python-pptx -q

print("✅ All packages installed.")

✅ All packages installed.


## Step 2: Configure LLM and Embedding Model
Uses your Gemini and (optionally) HuggingFace API keys stored in Colab Secrets.

In [2]:
import os
from llama_index.core import Settings
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.llms.gemini import Gemini
from google.colab import userdata

# --- Embedding Model ---
# all-MiniLM-L6-v2 is lightweight and captures semantic meaning well
Settings.embed_model = HuggingFaceEmbedding(model_name="all-MiniLM-L6-v2")

# --- LLM (Gemini) ---
GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
Settings.llm = Gemini(model="models/gemini-2.5-flash")

print("✅ LLM and embedding model configured.")

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/tmp/ipykernel_30979/3588214160.py:14: DeprecationWarning: Call to deprecated class Gemini. (Should use `llama-index-llms-google-genai` instead, using Google's latest unified SDK. See: https://docs.llamaindex.ai/en/stable/examples/llm/google_genai/This package will no longer be supported after version 0.6.2) -- Deprecated since version 0.6.2.
  Settings.llm = Gemini(model="models/gemini-2.5-flash")


✅ LLM and embedding model configured.


## Step 3: Extract Analysis Outputs into a Text Document

The RAG system needs text to index.  
We extract all **sheets outputs** (tables,key metrics etc) from your analysis workbook and save them as a plain-text file that LlamaIndex can load.

> **Upload** your `Analysis.xlsx files` when the file picker appears below.

In [3]:
import os
from google.colab import files
import pandas as pd # New import

os.makedirs("analysis_docs", exist_ok=True)

# --- Upload the Excel workbooks ---
print("Please upload your two Excel workbooks (e.g., data.xlsx, metadata.xlsx):")
uploaded = files.upload() # This allows multiple files to be selected in the UI

if len(uploaded) != 2:
    raise ValueError("Please upload exactly two Excel workbooks.")

excel_paths = list(uploaded.keys())
print(f"\n📓 Loaded Excel files: {', '.join(excel_paths)}")

# --- Extract outputs into structured text from Excel ---
def extract_excel_data(paths: list) -> str:
    """Read data from multiple Excel files and concatenate into a single text block."""
    all_sections = []
    for excel_path in paths:
        all_sections.append(f"--- START EXCEL WORKBOOK: {os.path.basename(excel_path)} ---")
        try:
            xls = pd.ExcelFile(excel_path)
            for sheet_name in xls.sheet_names:
                df = pd.read_excel(xls, sheet_name=sheet_name)
                all_sections.append(f"Sheet: {sheet_name}")
                # Convert DataFrame to markdown table format for readability
                all_sections.append(df.to_markdown(index=False))
                all_sections.append("\n") # Add a newline for separation between sheets
        except Exception as e:
            all_sections.append(f"Error reading {excel_path}: {e}")
        all_sections.append(f"--- END EXCEL WORKBOOK: {os.path.basename(excel_path)} ---\n")

    return "\n\n" + "=" * 60 + "\n\n".join(all_sections)


extracted_text = extract_excel_data(excel_paths) # Call the new function

output_doc_path = "analysis_docs/notebook_outputs.txt"
with open(output_doc_path, "w", encoding="utf-8") as f:
    f.write(extracted_text)

print(f"✅ Extracted {len(extracted_text):,} characters of analysis output from Excel files.")
print(f"   Saved to: {output_doc_path}")
print("\n--- Preview (first 1000 chars) ---")
print(extracted_text[:1000])


Please upload your two Excel workbooks (e.g., data.xlsx, metadata.xlsx):


Saving incident_text_analysis_outputs.xlsx to incident_text_analysis_outputs (6).xlsx
Saving incident_analysis_summary_tables.xlsx to incident_analysis_summary_tables (6).xlsx

📓 Loaded Excel files: incident_text_analysis_outputs (6).xlsx, incident_analysis_summary_tables (6).xlsx
✅ Extracted 94,215 characters of analysis output from Excel files.
   Saved to: analysis_docs/notebook_outputs.txt

--- Preview (first 1000 chars) ---


============================================================--- START EXCEL WORKBOOK: incident_text_analysis_outputs (6).xlsx ---

Sheet: Root Cause Classifications

| Incident ID   | Incident Type   | Supplier                  | Original Description                                                                                                                                                                                                                   | Original Gemini Category      | Final Root-Cause Category                  | Confidence   | Rule-Based

## Step 4: Build the RAG Index
Chunk, embed, and store the extracted analysis text in ChromaDB.

In [14]:
import chromadb
from llama_index.core import SimpleDirectoryReader, VectorStoreIndex, StorageContext
from llama_index.core.node_parser import SentenceSplitter
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.core.retrievers import VectorIndexRetriever
from llama_index.core.query_engine import RetrieverQueryEngine


# --- Load the extracted text document ---
print("Loading analysis document...")
documents = SimpleDirectoryReader("analysis_docs").load_data()
print(f"  Loaded {len(documents)} document(s).")

# --- Chunk the document ---
# Adjusted chunk_size and chunk_overlap for potential better context capture and flow
chunk_size=512 # balances context richness vs retrieval precision
chunk_overlap=100 # ensures no key insight is split across chunk boundaries
splitter = SentenceSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
nodes = splitter.get_nodes_from_documents(documents)
print(f"  Split into {len(nodes)} chunks.")

# --- Set up ChromaDB vector store ---
print("Initialising ChromaDB...")
db = chromadb.PersistentClient(path="./chroma_pptx_db")
chroma_collection = db.get_or_create_collection("pptx_rag_collection")
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)
storage_context = StorageContext.from_defaults(vector_store=vector_store)

# --- Build the index ---
print("Building vector index (this embeds all chunks)...")
index = VectorStoreIndex(nodes, storage_context=storage_context)

# --- Create a simple Vector Index Retriever ---
print("Creating Vector Index Retriever...")
retriever = VectorIndexRetriever(index=index, similarity_top_k=5)

# --- Create query engine with the simple retriever ---
print("Creating query engine...")
query_engine = RetrieverQueryEngine.from_args(
    retriever=retriever,
)

print("✅ RAG index and query engine ready (without hybrid search or re-ranking).")

Loading analysis document...
  Loaded 1 document(s).
  Split into 33 chunks.
Initialising ChromaDB...
Building vector index (this embeds all chunks)...
Creating Vector Index Retriever...
Creating query engine...
✅ RAG index and query engine ready (without hybrid search or re-ranking).


## Step 5: Define Slide Queries

Each slide maps to a targeted RAG query. The query fetches relevant context, which is then fed to Gemini to generate structured slide content (title, bullets, key stats).

You can customise the `SLIDE_QUERIES` list to match your problem's focus areas.

In [15]:
# Each entry defines one slide.
# 'query'        → sent to the RAG engine to retrieve relevant context
# 'slide_topic'  → used in the Gemini prompt to focus the slide content
# 'slide_type'   → hints the layout: 'title', 'bullets', 'stats', 'table'

SLIDE_QUERIES = [
    {
        "query": "What is the number of incidents per supplier? Provide this data in a format suitable for a visual representation, such as a table.",
        "slide_topic": "Incidents by Supplier",
        "slide_type": "table"
    },
    {
        "query": "What is the percentage of incidents per supplier compared to the total number of incidents? Provide this data in a format suitable for a visual representation, such as a table.",
        "slide_topic": "Incident Percentage by Supplier",
        "slide_type": "table"
    },
    {
        "query": "What is the total cost of resolving all incidents, and what is the average time required to resolve all incidents?",
        "slide_topic": "Incident Resolution: Cost & Time",
        "slide_type": "stats"
    },
    {
        "query": "Provide separate statistics for the incident duration for work orders and RMAs, including metrics like average, min, or max duration.",
        "slide_topic": "Incident Duration: Work Orders & RMAs",
        "slide_type": "bullets"
    },
    {
        "query": "Summarize the key insights and provide actionable recommendations derived from the incident analysis.",
        "slide_topic": "Key Insights & Recommendations",
        "slide_type": "bullets"
    }
]

print(f"✅ Defined {len(SLIDE_QUERIES)} slides to generate.")

✅ Defined 5 slides to generate.


## Step 6: RAG Query → Structured Slide Content

For each slide:
1. Retrieve relevant context from the vector store
2. Ask Gemini to synthesise the context into structured JSON (title, bullets, stats)
3. Store the result for PowerPoint assembly

In [16]:
import json as json_lib

def generate_slide_content(query_engine, slide_query: dict) -> dict:
    """
    1. Query the RAG engine to get relevant context from the notebook outputs.
    2. Use Gemini to turn that context into structured slide JSON.
    """
    query = slide_query["query"]
    topic = slide_query["slide_topic"]
    stype = slide_query["slide_type"]

    # --- RAG Retrieval ---
    try:
        rag_response = query_engine.query(query)
    except ValueError as e:
        if "Invalid fusion mode" in str(e):
            print("\nERROR: The RAG index configuration in 'index_cell' has an invalid fusion mode.\n")
            print("       Please modify the `index_cell` and ensure the `QueryFusionRetriever`'s `mode` parameter is set to 'reciprocal_rank' (instead of 'reciprocal_rank_fusion').\n")
            print("       Example: `mode='reciprocal_rank'`")
            raise e # Re-raise the original error after giving guidance
        else:
            raise e # Re-raise any other ValueErrors

    retrieved_context = rag_response.response

    # --- Gemini Synthesis Prompt ---
    synthesis_prompt = f"""You are a data analyst creating a professional PowerPoint presentation.

Slide Topic: {topic}
Slide Type: {stype}  (bullets = key points list; stats = big numbers + short labels; table = row/col data)

Retrieved context from the analysis:
{retrieved_context}

Generate the slide content as JSON with this exact structure:
{{
  "title": "<concise slide title>",
  "subtitle": "<optional one-line subtitle or leave empty string>",
  "bullets": ["<point 1>", "<point 2>", "<point 3>"],
  "stats": [
    {{"value": "<number or %>", "label": "<short label>"}},
    {{"value": "<number or %>", "label": "<short label>"}}
  ],
  "table_data": [
    ["Header 1", "Header 2"],
    ["Row 1 col 1", "Row 1 col 2"]
  ],
  "speaker_notes": "<2-3 sentences summarising the slide for the presenter>"
}}

Rules:
- bullets: 3-5 concise, insight-driven points (not vague generalities)
- stats: only include if there are real numbers in the context (2-4 stats max)
- table_data: If 'slide_type' is 'table', parse the markdown table from the 'retrieved_context' directly into this list of lists (including header row).
- Be specific – use actual numbers, categories, and findings from the context
- Return ONLY the JSON object, no markdown fences or extra text"""

    raw = Settings.llm.complete(synthesis_prompt).text.strip()

    # Strip markdown code fences if present
    raw = raw.replace("```json", "").replace("```", "").strip()

    try:
        slide_data = json_lib.loads(raw)
    except json_lib.JSONDecodeError:
        # Fallback: wrap raw text as a bullet slide
        slide_data = {
            "title": topic,
            "subtitle": "",
            "bullets": [raw[:200]],
            "stats": [],
            "table_data": [],
            "speaker_notes": raw[:300]
        }

    slide_data["slide_type"] = stype
    return slide_data


# --- Generate all slides ---
print("Generating slide content via RAG + Gemini...\n")
all_slides = []

for i, slide_query in enumerate(SLIDE_QUERIES, 1):
    print(f"  [{i}/{len(SLIDE_QUERIES)}] {slide_query['slide_topic']}...", end=" ")
    content = generate_slide_content(query_engine, slide_query)
    all_slides.append(content)
    print(f"✅  →  '{content['title']}'")

print(f"\n✅ All {len(all_slides)} slides generated.")

Generating slide content via RAG + Gemini...

  [1/5] Incidents by Supplier... ✅  →  'Incidents by Supplier'
  [2/5] Incident Percentage by Supplier... ✅  →  'Incident Percentage by Supplier'
  [3/5] Incident Resolution: Cost & Time... ✅  →  'Overall Incident Resolution Performance'
  [4/5] Incident Duration: Work Orders & RMAs... 

TooManyRequests: 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 5, model: gemini-2.5-flash
Please retry in 57.458326964s.

## Step 7: Assemble the PowerPoint

Uses `python-pptx` to build a professional deck with:
- A branded title slide
- Per-slide layouts adapted to content type (bullets / stats callouts)
- Speaker notes on every slide
- A closing summary slide

In [17]:
from pptx import Presentation
from pptx.util import Inches, Pt, Emu
from pptx.dml.color import RGBColor
from pptx.enum.text import PP_ALIGN
from pptx.util import Inches, Pt
import textwrap

# ── COLOUR PALETTE ──────────────────────────────────────────────────
DARK_BG   = RGBColor(0x1E, 0x27, 0x61)   # Deep navy  – title & closing slides
LIGHT_BG  = RGBColor(0xFF, 0xFF, 0xFF)   # White      – content slides
ACCENT    = RGBColor(0x00, 0x8B, 0xCC)   # Bright teal
TEXT_DARK = RGBColor(0x1E, 0x29, 0x3B)   # Near-black
TEXT_LIGHT= RGBColor(0xFF, 0xFF, 0xFF)   # White
MUTED     = RGBColor(0x64, 0x74, 0x8B)   # Grey
STAT_BG   = RGBColor(0xE0, 0xF2, 0xFE)   # Light blue card

# ── SLIDE DIMENSIONS (16:9) ─────────────────────────────────────────
W  = Inches(10)
H  = Inches(5.625)


# ────────────────────────────────────────────────────────────────────
# HELPER FUNCTIONS
# ────────────────────────────────────────────────────────────────────

def set_bg(slide, color: RGBColor):
    from pptx.oxml.ns import qn
    from lxml import etree
    bg = slide.background
    fill = bg.fill
    fill.solid()
    fill.fore_color.rgb = color


def add_text_box(slide, text, x, y, w, h,
                 font_size=18, bold=False, color=TEXT_DARK,
                 align=PP_ALIGN.LEFT, italic=False, wrap=True):
    txBox = slide.shapes.add_textbox(Inches(x), Inches(y), Inches(w), Inches(h))
    tf = txBox.text_frame
    tf.word_wrap = wrap
    p = tf.paragraphs[0]
    p.alignment = align
    run = p.add_run()
    run.text = text
    run.font.size = Pt(font_size)
    run.font.bold = bold
    run.font.italic = italic
    run.font.color.rgb = color
    return txBox


def add_accent_bar(slide, x, y, w=0.07, h=0.6, color=ACCENT):
    """Thin vertical bar used as a section marker beside a title."""
    from pptx.util import Inches
    shape = slide.shapes.add_shape(
        1,  # MSO_SHAPE_TYPE.RECTANGLE
        Inches(x), Inches(y), Inches(w), Inches(h)
    )
    shape.fill.solid()
    shape.fill.fore_color.rgb = color
    shape.line.fill.background()  # no border


def add_bullet_points(slide, bullets, x, y, w, h,
                      font_size=15, color=TEXT_DARK):
    txBox = slide.shapes.add_textbox(Inches(x), Inches(y), Inches(w), Inches(h))
    tf = txBox.text_frame
    tf.word_wrap = True
    for idx, bullet in enumerate(bullets):
        if idx == 0:
            para = tf.paragraphs[0]
        else:
            para = tf.add_paragraph()
        para.space_before = Pt(4)
        para.space_after  = Pt(4)
        run = para.add_run()
        run.text = f"▸  {bullet}"
        run.font.size = Pt(font_size)
        run.font.color.rgb = color


def add_stat_card(slide, value, label, cx, cy, cw=2.0, ch=1.3):
    """Blue rounded card with a big stat number and small label."""
    # Card background
    card = slide.shapes.add_shape(
        1, Inches(cx), Inches(cy), Inches(cw), Inches(ch)
    )
    card.fill.solid()
    card.fill.fore_color.rgb = STAT_BG
    card.line.color.rgb = ACCENT
    card.line.width = Pt(1.5)

    # Stat value
    add_text_box(
        slide, str(value),
        cx + 0.1, cy + 0.05, cw - 0.2, 0.65,
        font_size=28, bold=True, color=DARK_BG,
        align=PP_ALIGN.CENTER
    )
    # Label
    add_text_box(
        slide, label,
        cx + 0.05, cy + 0.7, cw - 0.1, 0.55,
        font_size=10, bold=False, color=MUTED,
        align=PP_ALIGN.CENTER
    )


def add_speaker_notes(slide, notes_text):
    if not notes_text:
        return
    notes_slide = slide.notes_slide
    tf = notes_slide.notes_text_frame
    tf.text = notes_text


# ────────────────────────────────────────────────────────────────────
# SLIDE BUILDERS
# ────────────────────────────────────────────────────────────────────

def build_title_slide(prs, title="Supplier Incident Data Analysis",
                       subtitle="GenAI-Powered Insights | WBS Group Assignment"):
    slide = prs.slides.add_slide(prs.slide_layouts[6])  # blank
    set_bg(slide, DARK_BG)

    # Accent strip on left
    strip = slide.shapes.add_shape(
        1, Inches(0), Inches(0), Inches(0.25), H
    )
    strip.fill.solid()
    strip.fill.fore_color.rgb = ACCENT
    strip.line.fill.background()

    # Title
    add_text_box(
        slide, title,
        0.6, 1.6, 8.8, 1.2,
        font_size=38, bold=True, color=TEXT_LIGHT,
        align=PP_ALIGN.LEFT
    )
    # Subtitle
    add_text_box(
        slide, subtitle,
        0.6, 2.9, 8.8, 0.8,
        font_size=18, bold=False, color=RGBColor(0xCA, 0xDC, 0xFC),
        align=PP_ALIGN.LEFT, italic=True
    )
    # Thin divider
    div = slide.shapes.add_shape(
        1, Inches(0.6), Inches(2.82), Inches(8), Inches(0.03)
    )
    div.fill.solid()
    div.fill.fore_color.rgb = ACCENT
    div.line.fill.background()

    # Footer
    add_text_box(
        slide, "Powered by RAG + Gemini 2.5 Flash",
        0.6, 4.9, 9, 0.5,
        font_size=10, color=MUTED, italic=True
    )
    return slide


def build_content_slide(prs, slide_data: dict):
    """Builds a bullets or stats content slide."""
    slide = prs.slides.add_slide(prs.slide_layouts[6])
    set_bg(slide, LIGHT_BG)

    title   = slide_data.get("title", "")
    bullets = slide_data.get("bullets", [])
    stats   = slide_data.get("stats", [])
    stype   = slide_data.get("slide_type", "bullets")
    notes   = slide_data.get("speaker_notes", "")

    # Top colour band
    band = slide.shapes.add_shape(
        1, Inches(0), Inches(0), W, Inches(1.0)
    )
    band.fill.solid()
    band.fill.fore_color.rgb = DARK_BG
    band.line.fill.background()

    # Slide title in band
    add_text_box(
        slide, title,
        0.35, 0.12, 9.0, 0.75,
        font_size=24, bold=True, color=TEXT_LIGHT
    )

    # Accent bar below band
    accent_bar = slide.shapes.add_shape(
        1, Inches(0), Inches(1.0), W, Inches(0.06)
    )
    accent_bar.fill.solid()
    accent_bar.fill.fore_color.rgb = ACCENT
    accent_bar.line.fill.background()

    # --- Stats layout ---
    if stype == "stats" and stats:
        # Up to 4 stat cards per row
        num_stats = min(len(stats), 4)
        card_w = 2.0
        total_w = num_stats * card_w + (num_stats - 1) * 0.3
        start_x = (10 - total_w) / 2

        for i, stat in enumerate(stats[:4]):
            cx = start_x + i * (card_w + 0.3)
            add_stat_card(slide, stat.get("value", ""), stat.get("label", ""), cx, 1.4)

        # Bullets below stats
        if bullets:
            add_bullet_points(slide, bullets, 0.5, 3.0, 9.0, 2.3)

    # --- Bullets layout ---
    else:
        add_bullet_points(slide, bullets, 0.5, 1.3, 9.0, 4.0)

    # Slide number (bottom right)
    add_text_box(
        slide, "WBS GenAI Assignment",
        0.3, 5.25, 5.0, 0.3,
        font_size=8, color=MUTED, italic=True
    )

    add_speaker_notes(slide, notes)
    return slide


def build_closing_slide(prs):
    slide = prs.slides.add_slide(prs.slide_layouts[6])
    set_bg(slide, DARK_BG)

    strip = slide.shapes.add_shape(
        1, Inches(0), Inches(0), Inches(0.25), H
    )
    strip.fill.solid()
    strip.fill.fore_color.rgb = ACCENT
    strip.line.fill.background()

    add_text_box(
        slide, "Thank You",
        0.6, 1.8, 8.8, 1.0,
        font_size=44, bold=True, color=TEXT_LIGHT,
        align=PP_ALIGN.LEFT
    )
    add_text_box(
        slide, "Questions & Discussion",
        0.6, 3.0, 8.8, 0.7,
        font_size=20, bold=False, color=RGBColor(0xCA, 0xDC, 0xFC),
        align=PP_ALIGN.LEFT, italic=True
    )
    return slide


# ────────────────────────────────────────────────────────────────────
# ASSEMBLE THE PRESENTATION
# ────────────────────────────────────────────────────────────────────

print("Assembling PowerPoint...")

prs = Presentation()
prs.slide_width  = W
prs.slide_height = H

# 1. Title slide
build_title_slide(
    prs,
    title="Supplier Incident Data Analysis",
    subtitle="GenAI-Powered Root Cause & Data Quality Insights  |  WBS Group Assignment"
)

# 2. Content slides
for slide_data in all_slides:
    build_content_slide(prs, slide_data)

# 3. Closing slide
build_closing_slide(prs)

OUTPUT_PATH = "GenAI_Group_Assignment_Presentation.pptx"
prs.save(OUTPUT_PATH)
print(f"✅ Presentation saved: {OUTPUT_PATH}  ({len(prs.slides)} slides)")

Assembling PowerPoint...
✅ Presentation saved: GenAI_Group_Assignment_Presentation.pptx  (5 slides)


## Step 8: Download the Presentation

In [18]:
from google.colab import files
files.download(OUTPUT_PATH)
print("📥 Download triggered — check your browser's downloads folder.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

📥 Download triggered — check your browser's downloads folder.


## Optional: Preview Slide Content
Print a text summary of all generated slides to verify the RAG retrieved sensible content before opening the file.

In [ ]:
for i, slide in enumerate(all_slides, 1):
    print(f"{'='*60}")
    print(f"SLIDE {i+1}: {slide.get('title', '')}")
    print(f"{'='*60}")

    if slide.get("bullets"):
        print("Bullets:")
        for b in slide["bullets"]:
            print(f"  ▸ {b}")

    if slide.get("stats"):
        print("Stats:")
        for s in slide["stats"]:
            print(f"  [{s.get('value')}]  {s.get('label')}")

    if slide.get("speaker_notes"):
        print(f"Notes: {slide['speaker_notes']}")
    print()

## Optional: Customise & Regenerate a Single Slide

If a slide's content isn't quite right, re-run the query with a different prompt — no need to regenerate everything.

In [ ]:
# ── Edit these two variables and run the cell ──────────────────────
SLIDE_INDEX  = 4        # 0-based index into all_slides
NEW_QUERY    = "Describe the GenAI classification model used, including the prompt design and confidence levels."
NEW_TOPIC    = "GenAI Classification Approach"
NEW_TYPE     = "bullets"   # 'bullets' or 'stats'
# ───────────────────────────────────────────────────────────────────

print(f"Regenerating slide {SLIDE_INDEX + 2} ('{NEW_TOPIC}')...")
new_content = generate_slide_content(
    query_engine,
    {"query": NEW_QUERY, "slide_topic": NEW_TOPIC, "slide_type": NEW_TYPE}
)
all_slides[SLIDE_INDEX] = new_content
print(f"✅ Slide updated: '{new_content['title']}'")
print("Re-run Step 7 (the pptx assembly cell) to rebuild the file.")